In [2]:
import os
import pandas as pd

import matplotlib.pyplot as plt
from numba.np.math.numbers import NAN

from data.visualize import plot_stations_on_dem, markers_from_stations_table



import peakweather

print("PeakWeather version ",peakweather.__version__)

from peakweather import PeakWeatherDataset

#Load Local Dataset
# Exclude: rain gauge stations (142) & sunshine variable
ds = PeakWeatherDataset(root="/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset",
                        parameters = ["temperature",
                                      "pressure",
                                      "humidity",
                                      "wind_speed",
                                      "wind_gust",
                                      "wind_direction",
                                      "precipitation",
                                      ],
                        extended_topo_vars="DEM",
                        imputation_method=None,
                        compute_uv = True,
                        station_type="meteo_station")

PeakWeather version  0.2.1


### Spatial Embedding

In [7]:
print("Check number of stations:", len(ds.stations_table))

Check number of stations: 156


In [ ]:
ds.stations_table.columns

In [ ]:
coordinates = ds.stations_table[['swiss_easting', 'swiss_northing']] # spatial embedding (lat, lon) → [sin(Bx), cos(Bx)]

#Call one station⁄
ds.stations_table.drop(columns=["latitude", "longitude", "station_name", "station_type"]).loc[station]

### Temporal Embedding

In [6]:
station = 'ABO'
ds.get_observations(station).head()

nat_abbr                         ABO                                        \
name                        humidity precipitation    pressure temperature   
datetime                                                                     
2017-01-01 00:00:00+00:00  30.500000           0.0  874.900024         1.1   
2017-01-01 00:10:00+00:00  30.700001           0.0  874.900024         1.2   
2017-01-01 00:20:00+00:00  34.599998           0.0  874.799988         1.2   
2017-01-01 00:30:00+00:00  27.900000           0.0  874.700012         2.0   
2017-01-01 00:40:00+00:00  32.000000           0.0  874.599976         0.5   

nat_abbr                                                                 \
name                      wind_direction wind_gust wind_speed    wind_u   
datetime                                                                  
2017-01-01 00:00:00+00:00          356.0       1.3        0.8  0.055805   
2017-01-01 00:10:00+00:00          277.0       1.8        1.0  0.992546   
2017-01-01 00:20:00+00:00          258.0       1.1        0.6  0.586889   
2017-01-01 00:30:00+00:00          306.0       1.5        0.9  0.728115   
2017-01-01 00:40:00+00:00          304.0       1.4        0.7  0.580326   

nat_abbr                             
name                         wind_v  
datetime                             
2017-01-01 00:00:00+00:00 -0.798051  
2017-01-01 00:10:00+00:00 -0.121869  
2017-01-01 00:20:00+00:00  0.124747  
2017-01-01 00:30:00+00:00 -0.529007  
2017-01-01 00:40:00+00:00 -0.391435

In [4]:
print("Start recording", ds.get_observations(station).index[0].date())
print("End recording", ds.get_observations(station).index[-1].date())

Start recording 2017-01-01
End recording 2025-10-13


In [ ]:
ds.get_observations(station).index[-1] - ds.get_observations(station).index[0]

In [ ]:
3207/365

In [ ]:
ds.get_observations(station).index

### Variables

In [ ]:
ds.get_observations(station).iloc[0]

In [ ]:
ds.available_parameters

#### Explore Variables Data

In [ ]:
ds_day = PeakWeatherDataset(root="/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset",
                        parameters = ["temperature",
                                      "pressure",
                                      "humidity",
                                      "wind_speed",
                                      "wind_gust",
                                      "wind_direction",
                                      "precipitation",
                                      ],
                        freq = "d",
                        station_type="meteo_station")

In [ ]:
df, mask = ds_day.get_observations(
    #first_date="2024-08-02 16:32",
    #last_date="2024-08-06 23:26",
    return_mask=True,
)

long_df = mask.stack(0).rename_axis(["datetime","nat_abbr"]).reset_index()
df_time = long_df.groupby(["datetime"]).sum().drop(columns=["nat_abbr"]).reset_index()

variables = df_station.columns[df_station.columns != 'nat_abbr']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for v in variables:
    ax.plot(
        df_time["datetime"],
        df_time[v].rolling(30).mean(),
        label=v
    )

ax.set_title("Station availability (Monthly average)")
ax.set_ylabel("Number of stations")
ax.set_xlabel("Datetime")

# legend on the right side
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

ax.grid(True)

plt.tight_layout()
plt.show()